In [1]:
import json
import numpy as np
import pandas as pd

In [2]:
impurity_precentage_cutoff = 85

In [3]:
# Load in refinement results from batch script
df_refres = pd.read_csv("batch_results.csv")

# uncode flagged because it is save as a dictionary not a string in the csv and we need the information in the {}
def decode_flagged(val):
    if pd.isna(val) or val == "":
        return None
    return json.loads(val)

df_refres["flagged"] = df_refres["flagged"].apply(decode_flagged)
is_flagged = df_refres["flagged"].apply(bool)

# get total length of the dataframe as this can be run as it's currently updating
total = len(df_refres)
# How many are too impure to be classified as a material that has successfully made P2/O3
high_impurity_count = (df_refres["impurity_percentage"] > impurity_precentage_cutoff).sum()
flagged_count = is_flagged.sum()

# Success mask meaning what compositions can we successfully get an adjusted composition from.
# The composition must have made a layer oxide in the first place (less that 90% impurity precentage). 
# It must have fully been refined (no flagged results) and it must have a useable P2/O3 ratio (not -1 although these compositions should be labelled as 100% impure as well.)
success_mask = (
    df_refres["adjusted_formula"].notna()
    & df_refres["p2_o3_ratio"].notna()
    & (df_refres["p2_o3_ratio"] != -1)
    & (df_refres["impurity_percentage"] < impurity_precentage_cutoff)
    & (df_refres["adjusted_formula"] != df_refres["original_formula"])
)
success_count = success_mask.sum()

print(f"Processed so far: {total}")
print(f">{impurity_precentage_cutoff}% impurity: {high_impurity_count} ({high_impurity_count / total * 100:.1f}%)")
print(f"Flagged: {flagged_count} ({flagged_count / total * 100:.1f}%)")
print(f"Usable adjusted compositions: {success_count} ({success_count / total * 100:.1f}%)")

Processed so far: 369
>85% impurity: 124 (33.6%)
Flagged: 113 (30.6%)
Usable adjusted compositions: 188 (50.9%)


In [4]:
df_refres[is_flagged].describe()

,impurity_percentage,p2_o3_ratio,rwp,threshold_for_multi_val,error
count,113.000000,113.000000,113.000000,113.0,0.0
mean,86.207876,0.594518,8.897345,3.0,NaN
std,10.967128,0.463853,2.476416,0.0,NaN
min,50.080000,-1.000000,4.610000,3.0,NaN
25%,78.410000,0.278937,7.340000,3.0,NaN
50%,86.880000,0.720062,8.500000,3.0,NaN
75%,96.860000,0.978643,9.610000,3.0,NaN
max,100.000000,1.000000,19.430000,3.0,NaN


In [5]:
# Overlap between flagged and high impurity
flagged_and_high_impurity = (is_flagged & (df_refres["impurity_percentage"] > impurity_precentage_cutoff)).sum()

if high_impurity_count > 0:
    print(f"Of the {high_impurity_count} with >{impurity_precentage_cutoff}% impurity, {flagged_and_high_impurity} are also flagged "
          f"({flagged_and_high_impurity / high_impurity_count * 100:.1f}%)")
if flagged_count > 0:
    print(f"Of the {flagged_count} flagged, {flagged_and_high_impurity} also have >90% impurity "
          f"({flagged_and_high_impurity / flagged_count * 100:.1f}%)")

Of the 124 with >85% impurity, 65 are also flagged (52.4%)
Of the 113 flagged, 65 also have >90% impurity (57.5%)


In [6]:
# Get experimental time and temperature results from Master List Experimental
df_exp_list = pd.read_csv("Master List Experimental.csv")
time_temp = df_exp_list[["UID", "Time", "Temperature"]]

# Merge it into df with refinement results
df_refres_tt = df_refres.merge(time_temp, left_on="uid", right_on="UID", how="left")
df_refres_tt

,uid,original_formula,adjusted_formula,impurity_percentage,p2_o3_ratio,rwp,flagged,background_removal,threshold_for_multi_val,dedupe_by_spacegroup,error,UID,Time,Temperature
0,U00001,NaCuO2,NaCuO2,94.92,0.143369,8.17,{},False,2.1,False,NaN,U00001,12,950
1,U00002,NaFeO2,NaFeO2,100.00,-1.000000,34.66,{},False,2.1,False,NaN,U00002,12,950
2,U00003,NaMnO2,NaMnO2,9.13,1.000000,21.42,{},False,2.1,False,NaN,U00003,12,950
3,U00004,NaNiO2,NaNiO2,97.89,0.000000,11.08,{},False,2.1,False,NaN,U00004,12,950
4,U00005,NaCu0.5Fe0.5O2,NaCu0.500646Fe0.499354O2,99.56,0.885714,10.13,{},False,2.1,False,NaN,U00005,12,950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
364,U00369,Na0.5Cu0.388889Mn0.388889Fe0.111111Ni0.111111O2,Na0.5Cu0.218038Fe0.181351Mn0.41926Ni0.181351O2,42.76,1.000000,8.97,{},False,2.1,False,NaN,U00369,3,950
365,U00339,Na0.75Cu0.388889Ni0.388889Fe0.111111Mn0.111111O2,NaN,88.41,1.000000,7.19,"{'Ni': 0.4817055595247953, 'Mn': 0.20847291798...",True,3.0,True,NaN,U00339,3,950
366,U00358,Na0.5Ni0.666667Fe0.111111Cu0.111111Mn0.111111O2,NaN,82.28,0.170377,8.66,{'Ni': 1.069342194118677},True,3.0,True,NaN,U00358,3,950
367,U00365,Na0.5Cu0.388889Ni0.388889Fe0.111111Mn0.111111O2,NaN,88.61,0.965265,6.43,"{'Fe': 0.15972543872263895, 'Ni': 0.4630517041...",True,3.0,True,NaN,U00365,3,950


In [7]:
# Cat_Success — independent of flagged status
unsuccessful_mask = (
    (df_refres_tt["impurity_percentage"] >= impurity_precentage_cutoff)
    | df_refres_tt["p2_o3_ratio"].isna()
    | (df_refres_tt["p2_o3_ratio"] == -1)
)
df_refres_tt["Cat_Success"] = np.where(unsuccessful_mask, "Unsuccessful", "Successful")

# U-rows: original experimental composition, always included
u_rows = pd.DataFrame({
    "UID": df_refres_tt["uid"],
    "Chemical": df_refres_tt["original_formula"],
    "Time": df_refres_tt["Time"],
    "Temperature": df_refres_tt["Temperature"],
    "P2O3ratio": df_refres_tt["p2_o3_ratio"],
    "Cat_Success": df_refres_tt["Cat_Success"],
    "Impurity_Percentage" : df_refres_tt["impurity_percentage"]
})

u_rows.loc[u_rows["Cat_Success"] == "Unsuccessful", "P2O3ratio"] = -1

In [8]:
# A-rows: adjusted composition — only when NOT flagged, an adjustment
# exists, it actually differs from the original, and the material is
# independently marked Successful

has_adjusted = (
    ~is_flagged
    & df_refres_tt["adjusted_formula"].notna()
    & (df_refres_tt["Cat_Success"] == "Successful")
    & (df_refres_tt["adjusted_formula"] != df_refres_tt["original_formula"])
)
adj_df = df_refres_tt[has_adjusted]

a_rows = pd.DataFrame({
    "UID": "A" + adj_df["uid"].str[1:],
    "Chemical": adj_df["adjusted_formula"],
    "Time": adj_df["Time"],
    "Temperature": adj_df["Temperature"],
    "P2O3ratio": adj_df["p2_o3_ratio"],
    "Cat_Success": adj_df["Cat_Success"],
    "Impurity_Percentage" : adj_df["impurity_percentage"]
})

In [9]:
u_lookup = u_rows.set_index(u_rows["UID"].str[1:])["Cat_Success"]
a_check = a_rows.copy()
a_check["u_counterpart_success"] = a_check["UID"].str[1:].map(u_lookup)
print((a_check["u_counterpart_success"] != "Successful").sum())  # should be 0

0


In [12]:
# Combine, sort, save
final_df = pd.concat([u_rows, a_rows], ignore_index=True)
final_df = final_df.sort_values("UID").reset_index(drop=True)
final_df.to_csv("adj_exp_dataset.csv", index=False)

final_df.head(15)

,UID,Chemical,Time,Temperature,P2O3ratio,Cat_Success,Impurity_Percentage
0,A00006,NaCu0.275073Mn0.724927O2,12,950,1.000000,Successful,43.56
1,A00008,NaFe0.497381Mn0.502619O2,12,950,0.997606,Successful,63.63
2,A00010,NaMn0.736534Ni0.263466O2,12,950,0.978199,Successful,31.24
3,A00011,NaCu0.250091Fe0.523806Mn0.226103O2,12,950,0.627143,Successful,41.06
4,A00013,NaCu0.3296Mn0.63802Ni0.03238O2,12,950,0.980703,Successful,56.44
5,A00014,NaFe0.04303Mn0.518076Ni0.438894O2,12,950,0.931643,Successful,71.18
6,A00015,NaCu0.218298Fe0.535474Mn0.141456Ni0.104772O2,12,950,0.771327,Successful,56.55
7,A00021,Na0.75Cu0.132539Mn0.867461O2,12,950,1.000000,Successful,42.48
8,A00023,Na0.75Fe0.478742Mn0.521258O2,12,950,0.957368,Successful,67.64
9,A00025,Na0.75Mn0.879347Ni0.120653O2,12,950,0.994215,Successful,68.01
